# Preparación de datos: muestra online de H&M (Lookwise Data Lab)

**Objetivo de este notebook:** prepararar el DATASET con el cual se trabajara posteriormente: solo compras online, con una muestra de clientes reproducible y guardada en Parquet.

**Decisiones tomadas**
- Se trabaja **solo con el canal online** (`sales_channel_id` = 2). 
- La muestra se sortea por **cliente** (con todo su historial online), con semilla fija.
- Se generan dos tamaños, **5% (p5)** y **10% (p10)**; el tamaño final se eligirá según la RAM de la compu más justa del equipo de trabajo.

**Índice**
1. Preparación del entorno
2. Verificación del canal de venta
3. Filtro online
4. Verificación de las tablas originales
5. Muestra de clientes con semilla
6. Guardado en Parquet
7. Controles de integridad de las muestras
8. Representatividad de las muestras
9. Conclusiones y pendientes

> Nota: en un notebook solo se muestra el resultado de la **última** consulta de cada celda.

> **Antes de correr el notebook:** hay que cambiar las rutas a las de tu compu. `ruta_trans` se define en la primera celda de código de la sección 2 y `ruta_base` en la celda de la sección 6 (guardado en Parquet). Las demás celdas usan esas variables.

## 1. Preparación del entorno

**Qué utilizamos:** instala DuckDB y pandas.
**Por qué:** DuckDB permite consultar los CSV grandes directamente (la tabla de transacciones tiene más de 31 millones de filas) y pandas sirve para ver los resultados como tablas.
**Nota:** en VS Code se usa `%pip` (no `!pip`) y después hay que reiniciar el kernel.

In [1]:
%pip install duckdb pandas

## 2. Verificación del canal de venta

El dataset no documenta qué significan los valores 1 y 2 de `sales_channel_id`. Antes de filtrar por online hay que verificarlo con los datos.

### 2.1 Transacciones por mes y canal (feb a jul 2020)

**Qué hace:** cuenta las transacciones de cada canal por mes, en el período del confinamiento.
**Por qué:** si un canal es la tienda física, debería caer cuando las tiendas cerraron (abril 2020).
**Nota:** la línea `ruta_trans = ruta_trans = ...` tiene la asignación repetida; es inofensiva y se puede dejar una sola.

In [2]:
import duckdb

ruta_trans = "RUTA/A/transactions_train.csv"

resultado = duckdb.sql(f"""
    SELECT strftime(CAST(t_dat AS DATE), '%Y-%m') AS mes,
           sales_channel_id,
           COUNT(*) AS n
    FROM '{ruta_trans}'
    WHERE t_dat BETWEEN '2020-02-01' AND '2020-07-31'
    GROUP BY mes, sales_channel_id
    ORDER BY mes, sales_channel_id
""").df()

print(resultado)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        mes  sales_channel_id        n
0   2020-02                 1   324660
1   2020-02                 2   677199
2   2020-03                 1   154087
3   2020-03                 2   893665
4   2020-04                 2  1340882
5   2020-05                 1   391746
6   2020-05                 2   970069
7   2020-06                 1   590345
8   2020-06                 2  1174162
9   2020-07                 1   535082
10  2020-07                 2   816420


**Resultado:** en abril 2020 el canal 1 **no aparece** en la tabla: no tiene ninguna transacción (no figura con 0, directamente no hay filas). El canal 2 aparece todos los meses.

**A tener en cuenta:**
- El canal 1 cae a menos de la mitad en marzo (de 324.660 a 154.087), desaparece en abril y vuelve en mayo.
- El canal 2 no se interrumpe y llega a su máximo en abril (1.340.882).
- Este patrón es coherente con el cierre de las tiendas físicas durante el confinamiento, por lo que **canal 1 = tienda y canal 2 = online**.
- Es una **inferencia**, no documentación oficial del dataset: la ausencia de registros también podría ser un hueco en los datos. Por eso se contrasta con el comportamiento de marzo y mayo y con los comentarios públicos sobre el dataset.

### 2.2 Porcentaje online en todo el período

**Qué hace:** cuenta transacciones por mes y canal en todo el dataset y calcula el % del canal 2.
**Por qué:** comprobar que el predominio del canal 2 no es solo un efecto de 2020.
**Resultado y conclusión:** en la celda de texto que sigue al resultado.

In [3]:
todo = duckdb.sql(f"""
    SELECT strftime(CAST(t_dat AS DATE), '%Y-%m') AS mes,
           sales_channel_id,
           COUNT(*) AS n
    FROM '{ruta_trans}'
    GROUP BY mes, sales_channel_id
    ORDER BY mes, sales_channel_id
""").df()

tabla = todo.pivot(index="mes", columns="sales_channel_id", values="n")
tabla["pct_online"] = (tabla[2] / (tabla[1] + tabla[2]) * 100).round(1)
print(tabla)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sales_channel_id         1          2  pct_online
mes                                              
2018-09           207326.0   387450.0        65.1
2018-10           422989.0   974051.0        69.7
2018-11           360007.0   910612.0        71.7
2018-12           442405.0   706422.0        61.5
2019-01           330749.0   932722.0        73.8
2019-02           376081.0   776331.0        67.4
2019-03           361536.0   925214.0        71.9
2019-04           442202.0  1034252.0        70.0
2019-05           490906.0  1069413.0        68.5
2019-06           565945.0  1340257.0        70.3
2019-07           578694.0  1228800.0        68.0
2019-08           403399.0   850131.0        67.8
2019-09           335011.0   892167.0        72.7
2019-10           353588.0   793184.0        69.2
2019-11           326721.0   871312.0        72.7
2019-12           434698.0   683617.0        61.1
2020-01           317316.0   759038.0        70.5
2020-02           324660.0   677199.0        67.6


**Resultado:** el canal 2 representa entre 60% y 74% de las transacciones de cada mes. Excepciones: marzo 2020 (85,3%) y abril 2020, donde el canal 1 no tiene registros y el porcentaje no se puede calcular (`NaN`).

## Verificación del canal de venta (`sales_channel_id`)

**Pregunta:** el dataset no documenta qué significa cada valor de `sales_channel_id` (1 y 2). Necesitamos saber cuál es online para filtrar la muestra.

**Fuentes previas (no oficiales):** comentarios públicos sobre el dataset indican 1 = tienda y 2 = online. Falta confirmar que el comentario sea del proveedor de los datos. Por eso lo verificamos con los datos.

### Consulta 1: transacciones por mes y canal (feb–jul 2020)

| mes | canal 1 | canal 2 |
|---|---|---|
| 2020-02 | 324.660 | 677.199 |
| 2020-03 | 154.087 | 893.665 |
| 2020-04 | — (sin transacciones) | 1.340.882 |
| 2020-05 | 391.746 | 970.069 |
| 2020-06 | 590.345 | 1.174.162 |
| 2020-07 | 535.082 | 816.420 |

**Lectura:** el canal 1 cae a menos de la mitad en marzo 2020, desaparece en abril y vuelve en mayo. El canal 2 nunca se interrumpe y llega a su máximo en abril. Ese patrón es coherente con el cierre de tiendas físicas por el confinamiento (dato de contexto, no del dataset).

### Consulta 2: porcentaje del canal 2 en todo el período (2018-09 a 2020-09)

- Entre sept 2018 y feb 2020, el canal 2 representa entre **61% y 74%** de las transacciones cada mes.
- Marzo 2020: **85,3%** (efecto del confinamiento). Abril 2020: el canal 1 no tiene registros.
- Desde mayo 2020 vuelve a **60%–71%**.

**Lectura:** el canal 2 es el mayoritario en todo el período (alrededor del 70%), no solo en 2020. Se mide en **cantidad de transacciones**, no en ingresos ni en clientes.

### Conclusión

- **Canal 2 = online, canal 1 = tienda.** Es una inferencia respaldada por el patrón de abril 2020 y por los comentarios públicos sobre el dataset. No es documentación oficial del dataset.
- Filtrar solo online conserva aproximadamente el 70% de las filas, así que la reducción fuerte del tamaño la hace el muestreo de clientes, no el filtro de canal.

### Pendientes / a tener en cuenta

- Marzo y abril 2020 son meses atípicos: decidir en el EDA si se dejan o se excluyen.
- Septiembre 2020 parece incompleto (250.404 en canal 1 y 547.865 en canal 2, contra 412.565 y 824.627 en agosto): revisar antes de usarlo en el test.
- Confirmar con el comentario original que la fuente sea el proveedor de los datos.

## 3. Filtro online

**Qué hace:** cuenta las filas y los clientes distintos del canal 2.
**Por qué:** es la base sobre la que se arma la muestra.

In [4]:
duckdb.sql(f"""
    SELECT COUNT(*) AS filas_online,
           COUNT(DISTINCT customer_id) AS clientes_online
    FROM '{ruta_trans}'
    WHERE sales_channel_id = 2
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,filas_online,clientes_online
0,22379862,1108418


**Resultado:** 22.379.862 filas online (alrededor del 70% de las 31.788.324 filas totales) y 1.108.418 clientes con al menos una compra online (alrededor del 81% de los 1.371.980 clientes).

**Conclusión:** filtrar online conserva la gran mayoría de los clientes y el 70% de las filas. La reducción fuerte de tamaño la hace el muestreo, no el filtro.

## 4. Verificación de las tablas originales

### 4.1 `articles`: estructura

**Qué hace:** define la ruta del archivo y muestra las primeras 5 filas.
**Por qué:** ver qué columnas describen a la prenda (tipo, color, sección, descripción), que son las que se usan en el enfoque por contenido.
**Nota:** solo se ve el resultado de la segunda consulta (`LIMIT 5`); el conteo de filas está en la celda siguiente.

In [5]:
ruta_art = "RUTA/A/articles.csv"

# 1) Cuántas filas trae
duckdb.sql(f"SELECT COUNT(*) AS filas FROM '{ruta_art}'").df()

# 2) Primeras filas y nombres de columnas
duckdb.sql(f"SELECT * FROM '{ruta_art}' LIMIT 5").df()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,09,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,0108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,09,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


**Resultado:** 25 columnas. `article_id` se lee como texto con los ceros adelante (por ejemplo `0108775015`).

**Conclusión:** hay atributos de producto suficientes para un modelo de contenido (`product_type_name`, `colour_group_name`, `section_name`, `detail_desc`, entre otros).

### 4.2 `articles`: claves únicas

**Qué hace:** compara la cantidad de filas con la de `article_id` distintos.
**Por qué:** si hubiera `article_id` repetidos, al unir con las transacciones se duplicarían filas.

In [6]:
duckdb.sql(f"""
    SELECT COUNT(*) AS filas,
           COUNT(DISTINCT article_id) AS articulos_unicos
    FROM '{ruta_art}'
""").df()

,filas,articulos_unicos
0,105542,105542


**Resultado:** 105.542 filas y 105.542 artículos únicos.

**Conclusión:** sin repetidos; se puede unir `articles` con `transactions` sin duplicar filas.

### 4.3 `customers`: filas y claves únicas

**Qué hace:** cuenta filas y `customer_id` distintos.
**Por qué:** misma razón que en `articles`.
**Nota:** la carpeta se llama `customer` (sin la "s").

In [7]:
ruta_cust = "RUTA/A/customers.csv"

duckdb.sql(f"""
    SELECT COUNT(*) AS filas,
           COUNT(DISTINCT customer_id) AS clientes_unicos
    FROM '{ruta_cust}'
""").df()

,filas,clientes_unicos
0,1371980,1371980


**Resultado:** 1.371.980 filas y 1.371.980 clientes únicos.

**Conclusión:** sin repetidos, y coincide con el total de clientes de las transacciones sin filtrar.

### 4.4 `customers`: valores vacíos

**Qué hace:** cuenta cuántos clientes tienen edad y código postal.
**Por qué:** la edad se usa para el baseline por segmento y el código postal para simular un mercado nuevo.
**Nota:** solo se ve la segunda consulta; la de `LIMIT 5` no se imprime.

In [8]:
# 1) Primeras filas
duckdb.sql(f"SELECT * FROM '{ruta_cust}' LIMIT 5").df()

# 2) Vacíos en edad y código postal
duckdb.sql(f"""
    SELECT COUNT(*) AS filas,
           COUNT(age) AS con_edad,
           COUNT(*) - COUNT(age) AS sin_edad,
           COUNT(postal_code) AS con_postal
    FROM '{ruta_cust}'
""").df()

,filas,con_edad,sin_edad,con_postal
0,1371980,1356119,15861,1371980


**Resultado:** 15.861 clientes sin edad (1,16%) y 0 sin código postal.

**Conclusión:** hay pocos vacíos, pero hay que decidir en el EDA qué recomendar a quienes no tienen edad (por ejemplo, el top global o un segmento "sin edad").

### 4.5 Vacíos en edad entre los clientes online

**Qué hace:** repite el conteo anterior, pero solo para los clientes que compraron por canal 2.
**Por qué:** la población de referencia del proyecto es la online, no todos los clientes.

In [9]:
duckdb.sql(f"""
    SELECT COUNT(*) AS clientes_online,
           COUNT(c.age) AS con_edad,
           COUNT(*) - COUNT(c.age) AS sin_edad,
           ROUND(100.0 * (COUNT(*) - COUNT(c.age)) / COUNT(*), 2) AS pct_sin_edad
    FROM '{ruta_cust}' c
    WHERE c.customer_id IN (
        SELECT DISTINCT customer_id
        FROM '{ruta_trans}'
        WHERE sales_channel_id = 2
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clientes_online,con_edad,sin_edad,pct_sin_edad
0,1108418,1096081,12337,1.11


**Resultado:** 1.108.418 clientes online (igual al conteo de la sección 3), con 12.337 sin edad (1,11%).

**Conclusión:** el filtro online no cambia el porcentaje de vacíos (1,16% contra 1,11%). La cifra de referencia para comparar con las muestras es 1,11%.

## 5. Muestra de clientes con semilla

**Qué hace:** a cada cliente online le asigna un número fijo, `bucket`, entre 0 y 9999, calculado con `hash(customer_id + semilla) % 10000`. La muestra **p5** son los clientes con `bucket < 500` y la **p10** los de `bucket < 1000`.
**Por qué:**
- Es **reproducible**: con la misma semilla (`lookwise42`) sale siempre lo mismo.
- La p5 queda **contenida** en la p10, así que se pueden probar varios tamaños sin rehacer nada.
- Se sortea por **cliente**, no por fila, para no cortar historiales.

In [10]:
SEMILLA = "lookwise42"

duckdb.sql(f"""
    CREATE OR REPLACE TABLE clientes_online AS
    SELECT customer_id,
           hash(customer_id || '{SEMILLA}') % 10000 AS bucket
    FROM (SELECT DISTINCT customer_id
          FROM '{ruta_trans}'
          WHERE sales_channel_id = 2)
""")

duckdb.sql("""
    SELECT COUNT(*) AS total,
           SUM(CASE WHEN bucket < 500  THEN 1 ELSE 0 END) AS p5,
           SUM(CASE WHEN bucket < 1000 THEN 1 ELSE 0 END) AS p10
    FROM clientes_online
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total,p5,p10
0,1108418,55454.0,110252.0


**Resultado:** 1.108.418 clientes online; p5 = 55.454 (5,00%) y p10 = 110.252 (9,95%).

**Conclusión:** los tamaños son los esperados. El 10% no da exactamente 10% porque el sorteo por número fijo es aleatorio.

### 5.1 Filas que tendría cada muestra

**Qué hace:** cuenta cuántas transacciones quedarían en cada tamaño, antes de guardar.
**Por qué:** estimar el peso de cada muestra.

In [11]:
duckdb.sql(f"""
    SELECT SUM(CASE WHEN c.bucket < 500  THEN 1 ELSE 0 END) AS filas_p5,
           SUM(CASE WHEN c.bucket < 1000 THEN 1 ELSE 0 END) AS filas_p10
    FROM '{ruta_trans}' t
    JOIN clientes_online c USING (customer_id)
    WHERE t.sales_channel_id = 2
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,filas_p5,filas_p10
0,1122274.0,2226486.0


**Resultado:** p5 = 1.122.274 filas (5,01% de las filas online) y p10 = 2.226.486 (9,95%).

**Conclusión:** el porcentaje de filas es casi igual al de clientes, lo que indica que el sorteo no se inclinó hacia clientes con mucha o poca actividad.

## 6. Guardado en Parquet

**Qué hace:** para cada tamaño crea una carpeta y guarda tres tablas en Parquet: `transactions` (solo canal 2 y solo clientes de la muestra), `customers` (solo los de la muestra) y `articles` (completa).
**Por qué:** Parquet pesa poco y se abre rápido, y se puede compartir por Drive.
**Nota:** las muestras se guardan en la carpeta `muestras_hm`.

In [12]:
import os

ruta_base = "RUTA/A/muestras_hm"
muestras = {"p5": 500, "p10": 1000}

for nombre, umbral in muestras.items():
    carpeta = f"{ruta_base}/{nombre}"
    os.makedirs(carpeta, exist_ok=True)
    ids = f"(SELECT customer_id FROM clientes_online WHERE bucket < {umbral})"

    duckdb.sql(f"""
        COPY (SELECT * FROM '{ruta_trans}'
              WHERE sales_channel_id = 2 AND customer_id IN {ids})
        TO '{carpeta}/transactions.parquet' (FORMAT PARQUET)
    """)
    duckdb.sql(f"""
        COPY (SELECT * FROM '{ruta_cust}' WHERE customer_id IN {ids})
        TO '{carpeta}/customers.parquet' (FORMAT PARQUET)
    """)
    duckdb.sql(f"""
        COPY (SELECT * FROM '{ruta_art}')
        TO '{carpeta}/articles.parquet' (FORMAT PARQUET)
    """)
    print(nombre, "listo")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

p5 listo


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

p10 listo


**Resultado:** imprime `p5 listo` y `p10 listo`.

**Conclusión:** que termine sin error no prueba que el contenido sea correcto; eso se controla en la sección siguiente.

## 7. Controles de integridad de las muestras

**Qué hace:** compara lo guardado con las cifras ya medidas: filas, clientes, filas de otros canales, claves huérfanas (transacciones sin cliente o sin artículo) y peso de cada archivo.

In [13]:
for nombre in ["p5", "p10"]:
    c = f"{ruta_base}/{nombre}"
    print(nombre)
    print(duckdb.sql(f"""
        SELECT
          (SELECT COUNT(*) FROM '{c}/transactions.parquet') AS filas_trans,
          (SELECT COUNT(DISTINCT customer_id) FROM '{c}/transactions.parquet') AS clientes_trans,
          (SELECT COUNT(*) FROM '{c}/customers.parquet') AS filas_cust,
          (SELECT COUNT(*) FROM '{c}/articles.parquet') AS filas_art,
          (SELECT COUNT(*) FROM '{c}/transactions.parquet' WHERE sales_channel_id <> 2) AS otros_canales,
          (SELECT COUNT(*) FROM '{c}/transactions.parquet'
             WHERE customer_id NOT IN (SELECT customer_id FROM '{c}/customers.parquet')) AS huerfanos_cliente,
          (SELECT COUNT(*) FROM '{c}/transactions.parquet'
             WHERE article_id NOT IN (SELECT article_id FROM '{c}/articles.parquet')) AS huerfanos_articulo
    """).df().T)
    for f in ["transactions", "customers", "articles"]:
        print(f, round(os.path.getsize(f"{c}/{f}.parquet") / 1e6, 1), "MB")

p5
                          0
filas_trans         1122274
clientes_trans        55454
filas_cust            55454
filas_art            105542
otros_canales             0
huerfanos_cliente         0
huerfanos_articulo        0
transactions 20.7 MB
customers 7.1 MB
articles 6.2 MB
p10
                          0
filas_trans         2226486
clientes_trans       110252
filas_cust           110252
filas_art            105542
otros_canales             0
huerfanos_cliente         0
huerfanos_articulo        0
transactions 42.6 MB
customers 14.1 MB
articles 6.2 MB


**Resultado:**

| Control | p5 | p10 |
|---|---|---|
| Filas de `transactions` | 1.122.274 | 2.226.486 |
| Clientes en `transactions` y en `customers` | 55.454 | 110.252 |
| Filas de `articles` | 105.542 | 105.542 |
| Filas de otros canales | 0 | 0 |
| Huérfanos (cliente / artículo) | 0 / 0 | 0 / 0 |
| Peso total aproximado | 34 MB | 63,5 MB |

**Conclusión:** todo coincide con lo esperado. Las tres tablas de cada muestra se pueden unir sin perder ni duplicar filas, y pesan poco.

## 8. Representatividad de las muestras

Que la muestra esté bien armada no garantiza que se parezca al total. Se compara con la población online completa.

### 8.1 Compras por cliente

**Qué hace:** calcula qué % de clientes tiene 1 compra, 2 a 5, 6 a 20 y más de 20.

In [14]:
import pandas as pd

def distribucion(origen):
    df = duckdb.sql(f"""
        SELECT CASE WHEN n = 1 THEN '1' WHEN n <= 5 THEN '2-5'
                    WHEN n <= 20 THEN '6-20' ELSE '>20' END AS tramo,
               COUNT(*) AS clientes
        FROM (SELECT customer_id, COUNT(*) AS n FROM {origen} GROUP BY customer_id)
        GROUP BY tramo
    """).df().set_index("tramo").reindex(["1", "2-5", "6-20", ">20"])
    return (100 * df["clientes"] / df["clientes"].sum()).round(2)

pd.DataFrame({
    "total_online": distribucion(f"'{ruta_trans}' WHERE sales_channel_id = 2"),
    "p5": distribucion(f"'{ruta_base}/p5/transactions.parquet'"),
    "p10": distribucion(f"'{ruta_base}/p10/transactions.parquet'"),
})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_online,p5,p10
tramo,,,
1,11.36,11.27,11.28
2-5,30.50,30.43,30.54
6-20,32.66,32.66,32.68
>20,25.47,25.63,25.49


**Resultado:** la diferencia máxima contra el total online es de 0,16 puntos (p5, tramo de más de 20).

**Conclusión:** las dos muestras son representativas en este aspecto.
**Dato para el EDA:** solo el 11% de los clientes online tiene una única compra y alrededor del 25% tiene más de 20. Los clientes "fríos" no están todos hechos en los datos: hay que construirlos recortando el historial, además de los clientes cuya primera compra cae después del corte.

### 8.2 Cobertura de artículos

**Qué hace:** cuenta cuántos artículos aparecen con al menos una compra y cuántos con 5 o más.
**Por qué:** con pocos clientes, muchos artículos quedan sin ventas, y eso limita sobre todo a los modelos de co-compra y de contenido.

In [15]:
def cobertura(origen):
    return duckdb.sql(f"""
        SELECT COUNT(*) AS articulos_vendidos,
               SUM(CASE WHEN n >= 5 THEN 1 ELSE 0 END) AS con_5_o_mas
        FROM (SELECT article_id, COUNT(*) AS n FROM {origen} GROUP BY article_id)
    """).df().iloc[0]

pd.DataFrame({
    "total_online": cobertura(f"'{ruta_trans}' WHERE sales_channel_id = 2"),
    "p5": cobertura(f"'{ruta_base}/p5/transactions.parquet'"),
    "p10": cobertura(f"'{ruta_base}/p10/transactions.parquet'"),
})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_online,p5,p10
articulos_vendidos,98322.0,68978.0,78318.0
con_5_o_mas,85223.0,38944.0,49820.0


**Resultado:**

| | Total online | p5 | p10 |
|---|---|---|---|
| Artículos con al menos 1 compra | 98.322 | 68.978 (70,2%) | 78.318 (79,7%) |
| Artículos con 5 o más compras | 85.223 | 38.944 (45,7%) | 49.820 (58,5%) |

**Conclusión:** la muestra es representativa en clientes pero no en cobertura de artículos. El 5% pierde alrededor del 30% de los artículos con ventas y el 10% alrededor del 20%. Es esperable al achicar la muestra. Lo que realmente importa es cuántas compras de la semana de test usan artículos que no están en el train; eso se mide una vez fijado el corte.

### 8.3 Fecha de primera compra

**Qué hace:** calcula qué % de los clientes hizo su primera compra online en cada mes.
**Por qué:** comprobar que la muestra no está sesgada hacia clientes nuevos o antiguos.

In [16]:
def primera_compra(origen):
    df = duckdb.sql(f"""
        SELECT strftime(primera, '%Y-%m') AS mes, COUNT(*) AS clientes
        FROM (SELECT customer_id, MIN(t_dat) AS primera FROM {origen} GROUP BY customer_id)
        GROUP BY mes ORDER BY mes
    """).df().set_index("mes")["clientes"]
    return (100 * df / df.sum()).round(2)

pd.DataFrame({
    "total_online": primera_compra(f"'{ruta_trans}' WHERE sales_channel_id = 2"),
    "p5": primera_compra(f"'{ruta_base}/p5/transactions.parquet'"),
    "p10": primera_compra(f"'{ruta_base}/p10/transactions.parquet'"),
})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_online,p5,p10
mes,,,
2018-09,7.06,6.94,6.93
2018-10,12.61,12.37,12.62
2018-11,9.66,9.76,9.72
2018-12,5.77,6.05,5.86
2019-01,5.43,5.36,5.40
2019-02,3.93,3.99,3.90
2019-03,3.86,3.82,3.79
2019-04,3.95,4.03,3.98
2019-05,3.90,4.04,4.03


**Resultado:** los meses de p5 y p10 siguen de cerca a los del total online. La mayor diferencia es de unos 0,3 puntos (p5, diciembre 2018).

**Conclusión:** las muestras son representativas también en antigüedad.
**A tener en cuenta:**
- Los primeros meses (sept y oct 2018) están inflados: la "primera compra" es la primera dentro de la ventana observada, y hay clientes que ya existían antes.
- Alrededor del 1,3% de los clientes online (unos 700 en p5) tiene su primera compra en septiembre 2020. Como septiembre parece incompleto, los clientes fríos naturales de la semana de test se miden cuando esté fijado el corte.

### 8.4 Clientes sin edad en las muestras

**Qué hace:** calcula el % de clientes sin edad en cada muestra.
**Por qué:** compararlo con la referencia de la población online (1,11%).

In [17]:
for n in ["p5", "p10"]:
    print(n)
    print(duckdb.sql(f"""
        SELECT COUNT(*) AS clientes,
               ROUND(100.0 * (COUNT(*) - COUNT(age)) / COUNT(*), 2) AS pct_sin_edad
        FROM '{ruta_base}/{n}/customers.parquet'
    """).df())

p5
   clientes  pct_sin_edad
0     55454          1.11
p10
   clientes  pct_sin_edad
0    110252           1.1


**Resultado:** p5 = 1,11% y p10 = 1,10%.

**Conclusión:** las muestras mantienen el porcentaje de vacíos de la población online.

## 9. Pruebas de tamaño: ¿aguanta la compu con el 10%?

Objetivo: decidir con números entre p5 y p10. Se miden cuatro cosas: **cobertura** de compras, **material de test**, **costo de memoria y tiempo** (benchmark) y una **prueba pesada**. Son controles de la muestra, no EDA.

**Qué hace:** para cada artículo cuenta cuántas veces se compró online y calcula qué porcentaje de *todas* las compras corresponde a artículos que aparecen en el p5 y en el p10.
**Por qué:** antes medimos cobertura contando artículos distintos. Pero un artículo con 2 compras pesa igual que uno con 10.000. Ponderar por compras muestra qué parte de la actividad real queda cubierta.

In [18]:
duckdb.sql(f"""
    WITH ventas AS (
        SELECT article_id, COUNT(*) AS n
        FROM '{ruta_trans}' WHERE sales_channel_id = 2
        GROUP BY article_id
    ),
    p5 AS (SELECT DISTINCT article_id FROM '{ruta_base}/p5/transactions.parquet'),
    p10 AS (SELECT DISTINCT article_id FROM '{ruta_base}/p10/transactions.parquet')
    SELECT
        ROUND(100.0 * SUM(CASE WHEN v.article_id IN (SELECT article_id FROM p5)  THEN n END) / SUM(n), 2) AS pct_compras_p5,
        ROUND(100.0 * SUM(CASE WHEN v.article_id IN (SELECT article_id FROM p10) THEN n END) / SUM(n), 2) AS pct_compras_p10
    FROM ventas v
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pct_compras_p5,pct_compras_p10
0,98.6,99.41


**Resultado:**

| | p5 | p10 |
|---|---|---|
| % de las compras online cubiertas por artículos de la muestra | 98,60% | 99,41% |

**Conclusión:** aunque el 5% pierde ~30% de los artículos distintos, esos son artículos de muy pocas ventas: las compras que quedan sin cubrir son 1,4% (p5) y 0,6% (p10). La cobertura ponderada es alta en ambas; el 10% deja menos de la mitad de lo que deja el 5%.

**Qué hace:** cuenta cuántos clientes distintos y cuántas compras hay desde el 15/09/2020 en cada muestra.
**Por qué:** la semana de test tiene que tener suficientes datos para evaluar. El 15/09/2020 es una fecha **provisoria** (los últimos 7 días del dataset); la fecha definitiva se fija al inicio del EDA con los conteos diarios, porque septiembre 2020 parece incompleto.

In [19]:
for n in ["p5", "p10"]:
    print(n)
    print(duckdb.sql(f"""
        SELECT COUNT(DISTINCT customer_id) AS clientes_con_compra,
               COUNT(*) AS compras
        FROM '{ruta_base}/{n}/transactions.parquet'
        WHERE t_dat >= '2020-09-15'
    """).df())

p5
   clientes_con_compra  compras
0                 2317     9317
p10
   clientes_con_compra  compras
0                 4600    18329


**Resultado:**

| | p5 | p10 |
|---|---|---|
| Clientes con compra desde el 15/09/2020 | 2.317 | 4.600 |
| Compras | 9.317 | 18.329 |

**Conclusión:** el p10 duplica el material de evaluación, como era de esperar. Ambas alcanzan para una métrica estable en general; la diferencia aparece en subgrupos chicos (ver clientes nuevos).

**Qué hace:** cuenta los clientes cuya **primera compra** en toda la muestra ocurre desde el 15/09/2020 (clientes nuevos reales, sin historia en el train).
**Por qué:** son los clientes en frío "de verdad", que es el foco del proyecto.

In [20]:
for n in ["p5", "p10"]:
    print(n)
    print(duckdb.sql(f"""
        SELECT COUNT(*) AS clientes_nuevos_en_test
        FROM (SELECT customer_id, MIN(t_dat) AS primera
              FROM '{ruta_base}/{n}/transactions.parquet'
              GROUP BY customer_id)
        WHERE primera >= '2020-09-15'
    """).df())

p5
   clientes_nuevos_en_test
0                      257
p10
   clientes_nuevos_en_test
0                      489


**Resultado:**

| | p5 | p10 |
|---|---|---|
| Clientes nuevos en la semana de test | 257 | 489 |

**Conclusión:** con el 5% hay 257 clientes fríos reales para evaluar; con el 10%, 489. El 5% no es inservible, pero queda justo para medir el caso en frío con segmentos (por ejemplo, 0, 1 o 2 compras visibles). Además se complementan con clientes fríos **simulados** (se oculta la historia de clientes con mucha historia).

**Qué hace:** muestra la RAM total y la disponible de la compu.
**Por qué:** el costo de usar el p10 depende de la memoria libre al momento de trabajar. Cada integrante debería correr esta celda en su compu.

In [21]:
import psutil
m = psutil.virtual_memory()
print(round(m.total / 1e9, 1), "GB en total;", round(m.available / 1e9, 1), "GB disponibles")

16.7 GB en total; 2.4 GB disponibles


**Resultado (compu de una integrante):** 16,7 GB de RAM en total y 3,0 GB disponibles en ese momento (con otras aplicaciones abiertas).

**Conclusión:** la memoria libre real es bastante menor que la total. Por eso el benchmark siguiente se mide con ese margen.

**Qué hace:** simula el trabajo típico de un baseline: carga las tres tablas del p10 desde Parquet, las une (transacciones + artículos + edad del cliente) y calcula el top 12 de artículos. Mide tiempos y el peso de la tabla unida en memoria.
**Por qué:** es el caso más cargado de la etapa de EDA y baseline (con las tres tablas unidas).

In [22]:
import time
import pandas as pd

base = f"{ruta_base}/p10"
t0 = time.time()
trans = pd.read_parquet(f"{base}/transactions.parquet")
cust = pd.read_parquet(f"{base}/customers.parquet")[["customer_id", "age"]]
art = pd.read_parquet(f"{base}/articles.parquet")[["article_id", "product_type_name", "colour_group_name", "section_name"]]
t1 = time.time()
df = trans.merge(art, on="article_id").merge(cust, on="customer_id")
t2 = time.time()
top = df.groupby("article_id").size().sort_values(ascending=False).head(12)
t3 = time.time()

print(f"Carga: {t1-t0:.1f} s | Uniones: {t2-t1:.1f} s | Top 12: {t3-t2:.1f} s | Total: {t3-t0:.1f} s")
print(f"Tabla unida: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")

Carga: 1.4 s | Uniones: 1.4 s | Top 12: 0.4 s | Total: 3.1 s
Tabla unida: 927 MB


**Resultado:** carga 1,7 s · uniones 1,5 s · top 12 0,3 s · **total 3,5 s** · tabla unida **927 MB**.

**Conclusión:** el p10 corre sin problema en una compu con 3 GB libres. La tabla unida ocupa menos de 1 GB.

**Qué hace:** prueba de estrés. Calcula todos los pares de artículos que compró un mismo cliente (co-compra), una operación típica de recomendadores por similitud y mucho más pesada que un top 12. Se limita DuckDB a 2 GB (`memory_limit`) para que no se cuelgue la compu.
**Por qué:** si el p10 aguanta esto, aguanta los usos previstos en el EDA. Se corre primero con el p5 y después con el p10 para ver cómo escala.

In [23]:
import time
import duckdb

duckdb.sql("SET memory_limit='2GB'")   # tope de seguridad para que no se cuelgue la compu

def prueba_pesada(nombre):
    t0 = time.time()
    r = duckdb.sql(f"""
        WITH compras AS (
            SELECT DISTINCT customer_id, article_id
            FROM '{ruta_base}/{nombre}/transactions.parquet'
        )
        SELECT COUNT(*) AS pares_distintos, SUM(n) AS co_ocurrencias
        FROM (
            SELECT a.article_id AS a1, b.article_id AS a2, COUNT(*) AS n
            FROM compras a
            JOIN compras b
              ON a.customer_id = b.customer_id AND a.article_id < b.article_id
            GROUP BY a1, a2
        )
    """).df()
    print(nombre, f"{time.time() - t0:.1f} s")
    return r

prueba_pesada("p5")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

p5 4.6 s


,pares_distintos,co_ocurrencias
0,28383189,31429744.0


In [24]:
prueba_pesada("p10")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

p10 12.1 s


,pares_distintos,co_ocurrencias
0,53149275,63074954.0


**Resultado:**

| | p5 | p10 |
|---|---|---|
| Tiempo | 4,7 s | 11,1 s |
| Pares distintos de artículos | 28.383.189 | 53.149.275 |
| Co-ocurrencias | 31.429.744 | 63.074.954 |

**Conclusión:** al duplicar la muestra el tiempo se multiplica por ~2,4 y los pares distintos casi por 1,9; terminó sin errores con el tope de 2 GB. Los pares repetidos (pares que aparecen en más de un cliente, que son los que aportan señal de co-compra) pasan de 3,05 M a 9,93 M (×3,3): el 10% aporta bastante más señal que el 5%, no solo más volumen.

## 10. Conclusiones y pendientes

**Conclusiones**
- El canal 2 es el online (inferencia respaldada por el patrón de abril 2020, no documentación oficial) y concentra ~70% de las transacciones. Todo el grupo confirmó trabajar solo con online.
- Se generaron dos muestras por cliente (hash determinístico, p5 ⊂ p10) con las tres tablas en Parquet, sin claves huérfanas ni otros canales.
- Ambas son representativas en compras por cliente, antigüedad de primera compra y vacíos de edad (diferencias de hasta 0,16 puntos en los tramos).
- **Cobertura:** por artículos distintos el 5% cubre 70% y el 10% 80%; ponderada por compras, 98,60% y 99,41%.
- **Test (desde el 15/09/2020, provisorio):** 2.317 vs 4.600 clientes; clientes nuevos reales 257 vs 489.
- **Costo:** el p10 se carga y une en 3,5 s (927 MB) y la prueba pesada tarda 11,1 s con tope de 2 GB.

**Decisión propuesta**
- Muestra oficial del proyecto: **p10**. Se confirma con el grupo corriendo el benchmark en cada compu **antes de empezar el EDA**.
- Todos usan **la misma** muestra. Si a alguien le falla, primero se prueba cerrar aplicaciones, cargar menos columnas o consultar el Parquet con DuckDB; recién si no alcanza se baja *todo el grupo* a p5.
- Una vez que arranca el EDA **no se cambia de tamaño**: cambiarlo después invalida la comparación de lo ya hecho. Si en el Sprint 2 un modelo pesa demasiado, se optimiza el código; el p5 (que está dentro del p10) puede usarse solo para pruebas rápidas, pero la evaluación final se hace sobre el p10.

**Pendientes**
- Que cada integrante corra el benchmark y el grupo confirme el tamaño.
- Fijar la fecha de corte con conteos diarios (septiembre 2020 parece incompleto) y verificar que 7 días de test sea razonable.
- Decidir en el EDA cómo tratar marzo–mayo 2020 (meses atípicos) y los clientes sin edad.
- Definir el método de clientes fríos simulados (se construye en Preparación de datos, con el corte fijado).
- Subir las carpetas a Drive y compartirlas con el grupo.